# Real-world Data Project: Retail Sales Analysis & Profit Prediction
**Domain:** Retail | **Tools:** Python, Pandas, Matplotlib, Seaborn, Scikit-learn

**Business question:** Which products, regions and pricing decisions drive profit, and can we predict whether an order will be profitable?

**Workflow:** load data -> inspect -> clean -> explore & visualise -> predictive model -> conclusions

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, classification_report

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
os.makedirs("visuals", exist_ok=True)
metrics = {}

## 1. Load and inspect the data

In [ ]:
df = pd.read_csv("data/retail_sales.csv")
metrics["raw_rows"] = len(df)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print()
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nExact duplicate rows:", df.duplicated().sum())
df.describe().round(2)

## 2. Data cleaning
Steps: remove duplicates, fill missing values (mode for categories, median for numbers), parse dates, and cap extreme outliers using the IQR rule.

In [ ]:
# 2.1 duplicates
metrics["duplicates_removed"] = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# 2.2 missing values
metrics["missing_filled"] = int(df.isna().sum().sum())
for col in ["Ship Mode", "Segment"]:
    df[col] = df[col].fillna(df[col].mode()[0])
for col in ["Quantity", "Discount"]:
    df[col] = df[col].fillna(df[col].median())

# 2.3 types
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Quantity"] = df["Quantity"].astype(int)

# 2.4 outliers (IQR with factor 3 = only extreme values), capped not deleted
q1, q3 = df["Sales"].quantile([0.25, 0.75])
upper = q3 + 3 * (q3 - q1)
mask = df["Sales"] > upper
metrics["outliers_capped"] = int(mask.sum())
ratio = upper / df.loc[mask, "Sales"]
df.loc[mask, "Profit"] = df.loc[mask, "Profit"] * ratio   # keep profit consistent with capped sales
df.loc[mask, "Sales"] = upper

metrics["clean_rows"] = len(df)
print(f"Rows: {metrics['raw_rows']} -> {metrics['clean_rows']}")
print(f"Duplicates removed: {metrics['duplicates_removed']} | Missing cells filled: {metrics['missing_filled']} | Outliers capped: {metrics['outliers_capped']}")
print("Remaining missing values:", int(df.isna().sum().sum()))

In [ ]:
# Before/after view of outlier handling
fig, ax = plt.subplots(figsize=(7, 3))
sns.boxplot(x=df["Sales"], ax=ax, color="#4C72B0")
ax.set_title("Sales distribution after outlier capping")
plt.tight_layout(); plt.savefig("visuals/00_sales_boxplot.png"); plt.show()

## 3. Feature engineering

In [ ]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["YearMonth"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["Margin"] = df["Profit"] / df["Sales"]
df["Profitable"] = (df["Profit"] > 0).astype(int)
df["Discount Band"] = pd.cut(df["Discount"], bins=[-0.01, 0, 0.2, 0.3, 0.5, 1],
                             labels=["0%", "1-20%", "21-30%", "31-50%", ">50%"])
metrics["total_sales"] = float(df["Sales"].sum())
metrics["total_profit"] = float(df["Profit"].sum())
metrics["overall_margin"] = metrics["total_profit"] / metrics["total_sales"]
metrics["loss_order_share"] = float(1 - df["Profitable"].mean())
print(f"Total sales: ${metrics['total_sales']:,.0f} | Total profit: ${metrics['total_profit']:,.0f} | Margin: {metrics['overall_margin']:.1%}")
print(f"Loss-making orders: {metrics['loss_order_share']:.1%}")

## 4. Exploratory analysis & visualisation

In [ ]:
# 4.1 Sales and profit trend
monthly = df.groupby("YearMonth")[["Sales", "Profit"]].sum()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly["Sales"], marker="o", label="Sales")
ax.plot(monthly.index, monthly["Profit"], marker="s", label="Profit")
ax.set_title("Monthly sales and profit"); ax.set_ylabel("USD"); ax.legend()
plt.tight_layout(); plt.savefig("visuals/01_monthly_trend.png"); plt.show()

seasonal = df.groupby("Month")["Sales"].sum()
metrics["peak_month"] = int(seasonal.idxmax())
metrics["peak_month_share"] = float(seasonal.max() / seasonal.sum())

In [ ]:
# 4.2 Category performance
cat_perf = df.groupby("Category").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
cat_perf["Margin"] = cat_perf["Profit"] / cat_perf["Sales"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cat_perf[["Sales", "Profit"]].plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_title("Sales and profit by category"); axes[0].set_ylabel("USD")
sns.barplot(x=cat_perf.index, y=cat_perf["Margin"] * 100, ax=axes[1], hue=cat_perf.index, legend=False)
axes[1].set_title("Profit margin by category (%)"); axes[1].set_ylabel("%"); axes[1].set_xlabel("")
plt.tight_layout(); plt.savefig("visuals/02_category_performance.png"); plt.show()
metrics["cat_perf"] = cat_perf.round(3).to_dict("index")

In [ ]:
# 4.3 Sub-category profit
sub_profit = df.groupby("Sub-Category")["Profit"].sum().sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#C44E52" if v < 0 else "#55A868" for v in sub_profit]
ax.barh(sub_profit.index, sub_profit.values, color=colors)
ax.set_title("Total profit by sub-category"); ax.set_xlabel("USD")
plt.tight_layout(); plt.savefig("visuals/03_subcategory_profit.png"); plt.show()
metrics["best_sub"] = (sub_profit.index[-1], float(sub_profit.iloc[-1]))
metrics["worst_sub"] = (sub_profit.index[0], float(sub_profit.iloc[0]))

In [ ]:
# 4.4 Region and segment
reg = df.groupby("Region").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
reg["Margin"] = reg["Profit"] / reg["Sales"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
reg["Sales"].sort_values().plot(kind="barh", ax=axes[0], color="#4C72B0"); axes[0].set_title("Sales by region")
sns.barplot(x=reg.index, y=reg["Margin"] * 100, ax=axes[1], hue=reg.index, legend=False)
axes[1].set_title("Profit margin by region (%)"); axes[1].set_ylabel("%"); axes[1].set_xlabel("")
plt.tight_layout(); plt.savefig("visuals/04_region_performance.png"); plt.show()
metrics["best_region"] = (reg["Margin"].idxmax(), float(reg["Margin"].max()))
metrics["worst_region"] = (reg["Margin"].idxmin(), float(reg["Margin"].min()))

In [ ]:
# 4.5 Discount impact (key business insight)
disc_perf = df.groupby("Discount Band", observed=True).agg(Orders=("Sales", "size"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
disc_perf["Margin"] = disc_perf["Profit"] / disc_perf["Sales"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=disc_perf.index.astype(str), y=disc_perf["Margin"] * 100, ax=axes[0], hue=disc_perf.index.astype(str), legend=False)
axes[0].axhline(0, color="black", lw=1); axes[0].set_title("Profit margin by discount band (%)"); axes[0].set_xlabel("Discount band"); axes[0].set_ylabel("%")
sample = df.sample(1500, random_state=0)
sns.scatterplot(data=sample, x="Discount", y="Profit", hue="Category", alpha=0.5, ax=axes[1])
axes[1].axhline(0, color="black", lw=1); axes[1].set_title("Discount vs profit per order")
plt.tight_layout(); plt.savefig("visuals/05_discount_impact.png"); plt.show()
heavy = df[df["Discount"] >= 0.4]
metrics["heavy_disc_share"] = float(len(heavy) / len(df))
metrics["heavy_disc_profit"] = float(heavy["Profit"].sum())
metrics["breakeven_note"] = disc_perf["Margin"].round(3).to_dict()

In [ ]:
# 4.6 Correlation heatmap
corr = df[["Sales", "Quantity", "Discount", "Profit", "Margin"]].corr()
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix")
plt.tight_layout(); plt.savefig("visuals/06_correlation.png"); plt.show()
metrics["corr_disc_margin"] = float(corr.loc["Discount", "Margin"])

## 5. Predictive modelling: will an order be profitable?
We predict `Profitable` (1 = profit above 0) from order attributes, comparing a Logistic Regression baseline with a Random Forest.

In [ ]:
features = ["Sales", "Quantity", "Discount", "Month", "Category", "Sub-Category", "Region", "Segment", "Ship Mode"]
X = pd.get_dummies(df[features], drop_first=True)
y = df["Profitable"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

baseline = max(y_test.mean(), 1 - y_test.mean())
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1).fit(X_train, y_train)

results = {}
for name, model in [("Logistic Regression", logreg), ("Random Forest", rf)]:
    pred = model.predict(X_test); proba = model.predict_proba(X_test)[:, 1]
    results[name] = {"accuracy": accuracy_score(y_test, pred), "auc": roc_auc_score(y_test, proba)}
metrics["baseline_acc"] = float(baseline)
metrics["models"] = {k: {m: float(v) for m, v in r.items()} for k, r in results.items()}
print(f"Majority-class baseline accuracy: {baseline:.3f}")
print(pd.DataFrame(results).T.round(3))
print()
print(classification_report(y_test, rf.predict(X_test), target_names=["Loss", "Profit"]))

In [ ]:
# Confusion matrix + ROC curve
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_test, rf.predict(X_test)), display_labels=["Loss", "Profit"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Random Forest: confusion matrix")
for name, model in [("Logistic Regression", logreg), ("Random Forest", rf)]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={results[name]['auc']:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1); axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curves"); axes[1].legend(loc="lower right")
plt.tight_layout(); plt.savefig("visuals/07_model_evaluation.png"); plt.show()

In [ ]:
# Feature importance
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=imp.values, y=imp.index, ax=ax, hue=imp.index, legend=False)
ax.set_title("Top 10 features (Random Forest)"); ax.set_xlabel("Importance")
plt.tight_layout(); plt.savefig("visuals/08_feature_importance.png"); plt.show()
metrics["top_features"] = [(k, float(v)) for k, v in imp.head(3).items()]

## 6. Key findings and recommendations

In [ ]:
import calendar
m = metrics
cp = m["cat_perf"]
best_cat = max(cp, key=lambda c: cp[c]["Margin"]); worst_cat = min(cp, key=lambda c: cp[c]["Margin"])
best_model = max(m["models"], key=lambda k: m["models"][k]["accuracy"])
findings = [
    f"Total sales were ${m['total_sales']:,.0f} with ${m['total_profit']:,.0f} profit, an overall margin of {m['overall_margin']:.1%}.",
    f"{calendar.month_name[m['peak_month']]} is the strongest month, contributing {m['peak_month_share']:.1%} of all sales (seasonal peak).",
    f"{best_cat} has the best margin ({cp[best_cat]['Margin']:.1%}); {worst_cat} has the weakest ({cp[worst_cat]['Margin']:.1%}).",
    f"Sub-category '{m['best_sub'][0]}' earns the most profit (${m['best_sub'][1]:,.0f}); '{m['worst_sub'][0]}' earns the least (${m['worst_sub'][1]:,.0f}).",
    f"Region {m['best_region'][0]} has the best margin ({m['best_region'][1]:.1%}); {m['worst_region'][0]} the weakest ({m['worst_region'][1]:.1%}).",
    f"Discounts of 40% or more make up {m['heavy_disc_share']:.1%} of orders and produced a net profit of {'-' if m['heavy_disc_profit'] < 0 else ''}${abs(m['heavy_disc_profit']):,.0f} in total (a loss); discount and margin correlate at {m['corr_disc_margin']:.2f}.",
    f"{best_model} predicts whether an order is profitable with {m['models'][best_model]['accuracy']:.1%} accuracy (baseline {m['baseline_acc']:.1%}) and AUC {m['models'][best_model]['auc']:.3f}.",
    f"The most influential features were: {', '.join(k for k, _ in m['top_features'])}.",
]
recommendations = [
    "Cap discounts at roughly 30% and require approval above that level, since deeper discounts erase the margin.",
    f"Review pricing and discounting in the weakest segments ({worst_cat}, {m['worst_region'][0]} region, and '{m['worst_sub'][0]}').",
    f"Build inventory and marketing capacity ahead of {calendar.month_name[m['peak_month']]} and the Q4 season.",
    "Use the profit-prediction model as a pre-approval check: flag discounted orders likely to lose money before they are confirmed.",
]
print("KEY FINDINGS")
for i, f in enumerate(findings, 1): print(f"{i}. {f}")
print("\nRECOMMENDATIONS")
for i, r in enumerate(recommendations, 1): print(f"{i}. {r}")

## 7. Limitations and next steps
- Results describe the supplied dataset only; correlation does not prove causation.
- Predictions use order-level attributes; adding customer history and product cost data would improve them.
- Next steps: tune hyperparameters with cross-validation, try Gradient Boosting, and build a Power BI / Tableau dashboard.